# 06c Broaching RGB Baseline Comparison – Fixed-Split Evaluation

This notebook evaluates the classical anomaly-detection baselines on the
industrial broaching dataset using the full three-channel RGB representation.

All models are fitted exclusively on the fixed nominal training set.
Decision thresholds are calibrated on the labeled validation set and
subsequently applied unchanged to the independent test set.

The evaluation protocol therefore matches the primary fixed-split
evaluation used for the CNN autoencoder.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

In [2]:
CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'data').exists() else CWD.parent

BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'

IMAGE_SIZE = (150, 100)
PCA_COMPONENTS = 32
SEED = 67

BROACH_MANIFEST = (
    REPO_ROOT
    / 'reports'
    / 'manifests'
    / 'broach_dataset_split_seed42.csv'
)

DATASETS = [

    {
        'name': 'broach_dataset',
        'manifest_path': BROACH_MANIFEST,
        'source_dataset': 'broach_dataset',
    },
]

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True) 

In [3]:
def load_broach_manifest(config: dict) -> pd.DataFrame:
    manifest = pd.read_csv(config["manifest_path"])

    manifest = manifest.copy()
    manifest["dataset"] = "broach_dataset"
    manifest["sample_id"] = (
        manifest["filename"]
        .str.replace(".png", "", regex=False)
    )

    manifest["target"] = manifest["label"]

    manifest["label"] = manifest["target"].map({
        0: "normal",
        1: "anomaly"
    })

    manifest["image_path"] = manifest.apply(
        lambda row:
            f"data/03_broach_dataset/train/{row['filename']}"
            if row["split"] == "train"
            else f"data/03_broach_dataset/test/{row['filename']}",
        axis=1,
    )

    return manifest[
        [
            "dataset",
            "sample_id",
            "split",
            "label",
            "target",
            "image_path",
        ]
    ]


def build_dataset_manifest(config: dict) -> pd.DataFrame:

    if config["name"] == "broach_dataset":
        return load_broach_manifest(config)

    raise ValueError(f"Unknown dataset: {config['name']}")


In [4]:
dataset_manifests = {config['name']: build_dataset_manifest(config) for config in DATASETS}
for dataset_name, dataset_manifest in dataset_manifests.items():
    print(dataset_name)
    display(dataset_manifest.groupby(['split', 'label']).size().rename('n').reset_index())

broach_dataset


,split,label,n
0,test,anomaly,22
1,test,normal,2478
2,train,normal,5000
3,validation,anomaly,23
4,validation,normal,2477


In [5]:
def open_resized_image(image_path: Path, mode: str) -> np.ndarray:
    image = Image.open(image_path).convert(mode).resize(IMAGE_SIZE)
    return np.asarray(image, dtype='float32') / 255.0


def image_vector(image_path: Path) -> np.ndarray:
    return open_resized_image(image_path, 'RGB').reshape(-1)


def channel_descriptor(arr: np.ndarray) -> np.ndarray:
    grad_y, grad_x = np.gradient(arr)

    hist, _ = np.histogram(
        arr,
        bins=32,
        range=(0.0, 1.0),
        density=True,
    )

    percentiles = np.percentile(
        arr,
        [1, 5, 25, 50, 75, 95, 99],
    )

    summary = np.array([
        arr.mean(),
        arr.std(),
        arr.min(),
        arr.max(),
        np.abs(grad_x).mean(),
        np.abs(grad_y).mean(),
        np.sqrt(grad_x ** 2 + grad_y ** 2).mean(),
    ], dtype='float32')

    return np.concatenate([
        summary,
        percentiles.astype('float32'),
        hist.astype('float32'),
        arr.mean(axis=0),
        arr.mean(axis=1),
        arr.std(axis=0),
        arr.std(axis=1),
    ]).astype('float32')

def image_descriptor(image_path: Path) -> np.ndarray:
    arr = open_resized_image(image_path, 'RGB')

    channel_features = [
        channel_descriptor(arr[:, :, channel])
        for channel in range(3)
    ]
    return np.concatenate(channel_features).astype('float32')


def load_matrix(rows: pd.DataFrame, extractor) -> np.ndarray:
    values = [extractor(REPO_ROOT / image_path) for image_path in rows['image_path']]
    if not values:
        raise ValueError('No images loaded.')
    return np.stack(values, axis=0)

In [6]:
def select_best_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        threshold = float(np.max(scores))
        return {
            'threshold': threshold,
            'validation_f1': 0.0,
            'validation_precision': 0.0,
            'validation_recall': 0.0,
        }

    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {
        'threshold': float(thresholds[best_idx]),
        'validation_f1': float(f1[best_idx]),
        'validation_precision': float(precision[best_idx]),
        'validation_recall': float(recall[best_idx]),
    }


def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def append_score_frame(score_frames: list[pd.DataFrame], rows: pd.DataFrame, method: str, score_values: np.ndarray) -> None:
    score_frames.append(pd.DataFrame({
        'dataset': rows['dataset'].to_numpy(),
        'method': method,
        'sample_id': rows['sample_id'].to_numpy(),
        'split': rows['split'].to_numpy(),
        'label': rows['label'].to_numpy(),
        'target': rows['target'].to_numpy(),
        'score_value': score_values,
    }))

In [10]:
# fit descriptor models
def fit_descriptor_baselines(
    train_nominal: pd.DataFrame,
):
    X_train = load_matrix(
        train_nominal,
        image_descriptor,
    )

    scaler = StandardScaler().fit(
        X_train
    )

    X_train_scaled = scaler.transform(
        X_train
    )

    ocsvm = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05,
    ).fit(
        X_train_scaled
    )

    iforest = IsolationForest(
        random_state=SEED,
        contamination="auto",
    ).fit(
        X_train_scaled
    )

    return scaler, ocsvm, iforest 

In [11]:
# use descriptor models
def score_descriptor_baselines(
    scaler,
    ocsvm,
    iforest,
    evaluation_fold: pd.DataFrame,
):
    X_eval = load_matrix(
        evaluation_fold,
        image_descriptor,
    )

    X_eval = scaler.transform(
        X_eval
    )

    return {
        "one_class_svm_rgb_features":
            -ocsvm.decision_function(X_eval),

        "isolation_forest_rgb_features":
            -iforest.decision_function(X_eval),
    }

In [12]:
# fit pca descriptor
def fit_pca_baseline(
    train_nominal: pd.DataFrame,
):
    X_train = load_matrix(
        train_nominal,
        image_vector,
    )

    n_components = min(
        PCA_COMPONENTS,
        X_train.shape[0],
        X_train.shape[1],
    )

    pca = PCA(
        n_components=n_components,
        random_state=SEED,
        svd_solver="randomized",
    ).fit(
        X_train
    )

    return pca

In [13]:
# use pca
def score_pca_baseline(
    pca,
    evaluation_fold: pd.DataFrame,
):
    X_eval = load_matrix(
        evaluation_fold,
        image_vector,
    )

    recon = pca.inverse_transform(
        pca.transform(X_eval)
    )

    return np.mean(
        (X_eval - recon) ** 2,
        axis=1,
    )

In [16]:
# ============================================================
# Fixed train / validation / test split
# ============================================================

broaching_manifest=dataset_manifests["broach_dataset"]

train_nominal = (
    broaching_manifest[
        (broaching_manifest["split"] == "train")
        & (broaching_manifest["target"] == 0)
    ]
    .copy()
    .reset_index(drop=True)
)

validation = (
    broaching_manifest[
        broaching_manifest["split"] == "validation"
    ]
    .copy()
    .reset_index(drop=True)
)

test = (
    broaching_manifest[
        broaching_manifest["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(train_nominal) == 5000
assert set(validation["target"].unique()) == {0, 1}
assert set(test["target"].unique()) == {0, 1}
assert (train_nominal["target"] == 0).all()

print("Broaching fixed split")
print("---------------------")
print(f"Nominal training samples: {len(train_nominal)}")
print(f"Validation samples:       {len(validation)}")
print(f"Test samples:             {len(test)}")

print("\nValidation balance:")
print(validation["label"].value_counts())

print("\nTest balance:")
print(test["label"].value_counts())

Broaching fixed split
---------------------
Nominal training samples: 5000
Validation samples:       2500
Test samples:             2500

Validation balance:
label
normal     2477
anomaly      23
Name: count, dtype: int64

Test balance:
label
normal     2478
anomaly      22
Name: count, dtype: int64


In [17]:
# ============================================================
# Fit RGB baselines on nominal training data
# ============================================================

scaler, ocsvm, iforest = (
    fit_descriptor_baselines(
        train_nominal
    )
)

pca = fit_pca_baseline(
    train_nominal
)

In [18]:
# ============================================================
# Validation scores
# ============================================================

y_val = validation[
    "target"
].to_numpy()

descriptor_val = (
    score_descriptor_baselines(
        scaler,
        ocsvm,
        iforest,
        validation,
    )
)

pca_val = score_pca_baseline(
    pca,
    validation,
)

validation_scores = {
    **descriptor_val,
    "pca_rgb_reconstruction": pca_val,
}

In [20]:
# ============================================================
# Decision-threshold calibration
# ============================================================

thresholds = {}

for method, scores in validation_scores.items():

    selected = select_best_f1_threshold(
        y_val,
        scores,
    )

    thresholds[method] = selected

    print(
        f"{method}\n"
        f"  Threshold: "
        f"{selected['threshold']:.6f}\n"
        f"  Validation F1: "
        f"{selected['validation_f1']:.4f}\n"
    )

one_class_svm_rgb_features
  Threshold: 14.167647
  Validation F1: 0.7391

isolation_forest_rgb_features
  Threshold: 0.152294
  Validation F1: 0.5556

pca_rgb_reconstruction
  Threshold: 0.000668
  Validation F1: 0.8444



In [21]:
# ============================================================
# Independent test-set scoring
# ============================================================

y_test = test[
    "target"
].to_numpy()

descriptor_test = (
    score_descriptor_baselines(
        scaler,
        ocsvm,
        iforest,
        test,
    )
)

pca_test = score_pca_baseline(
    pca,
    test,
)

test_scores = {
    **descriptor_test,
    "pca_rgb_reconstruction": pca_test,
}

In [22]:
# ============================================================
# Final fixed-split evaluation
# ============================================================

metric_rows = []
score_frames = []

for method, scores in test_scores.items():

    threshold = thresholds[
        method
    ]["threshold"]

    metrics = evaluate_at_threshold(
        y_test,
        scores,
        threshold,
    )

    score_name = (
        "reconstruction_mse"
        if method == "pca_rgb_reconstruction"
        else "anomaly_score"
    )

    metric_rows.append({
        "dataset": "broach_dataset",
        "method": method,
        "score": score_name,
        "threshold": threshold,
        **metrics,
    })

    append_score_frame(
        score_frames,
        test,
        method,
        scores,
    )


broaching_metrics = pd.DataFrame(
    metric_rows
)

broaching_scores = pd.concat(
    score_frames,
    ignore_index=True,
)

display(
    broaching_metrics
)

,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,broach_dataset,one_class_svm_rgb_features,anomaly_score,14.167647,0.838384,0.700000,0.777778,0.636364,2474,4,8,14
1,broach_dataset,isolation_forest_rgb_features,anomaly_score,0.152294,0.640653,0.530612,0.481481,0.590909,2464,14,9,13
2,broach_dataset,pca_rgb_reconstruction,reconstruction_mse,0.000668,0.982175,0.842105,1.000000,0.727273,2478,0,6,16


In [23]:
# ============================================================
# Export Broaching RGB fixed-split baseline results
# ============================================================

METRICS_PATH = (
    TABLE_DIR
    / "metrics_broaching_baselines_fixedsplit_rgb.csv"
)

SCORES_PATH = (
    BASELINE_DIR
    / "baseline_scores_broaching_fixedsplit_rgb.csv"
)

THRESHOLDS_PATH = (
    THRESHOLD_DIR
    / "baseline_thresholds_broaching_fixedsplit_rgb.json"
)


# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

broaching_metrics.to_csv(
    METRICS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Save test-set anomaly scores
# ------------------------------------------------------------

broaching_scores.to_csv(
    SCORES_PATH,
    index=False,
)


# ------------------------------------------------------------
# Save validation-calibrated thresholds
# ------------------------------------------------------------

with THRESHOLDS_PATH.open("w") as f:
    json.dump(
        thresholds,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("Wrote:")
print(f"  {METRICS_PATH}")
print(f"  {SCORES_PATH}")
print(f"  {THRESHOLDS_PATH}")

display(broaching_metrics)

Wrote:
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broaching_baselines_fixedsplit_rgb.csv
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\baselines\baseline_scores_broaching_fixedsplit_rgb.csv
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\thresholds\baseline_thresholds_broaching_fixedsplit_rgb.json


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,broach_dataset,one_class_svm_rgb_features,anomaly_score,14.167647,0.838384,0.700000,0.777778,0.636364,2474,4,8,14
1,broach_dataset,isolation_forest_rgb_features,anomaly_score,0.152294,0.640653,0.530612,0.481481,0.590909,2464,14,9,13
2,broach_dataset,pca_rgb_reconstruction,reconstruction_mse,0.000668,0.982175,0.842105,1.000000,0.727273,2478,0,6,16
